In [ ]:
%load_ext tensorboard
#!rm -rf ./logs/ 

import os
import pandas as pd
import numpy as np
import tensorflow as tf

from datetime import datetime
from tensorflow import keras
from tensorflow.keras import layers
from tensorboard.plugins.hparams import api as hp

import warnings
warnings.filterwarnings("ignore")

print(tf.version.VERSION)
print(keras.__version__)

gpus = tf.config.list_physical_devices("GPU")        
log_gpus = tf.config.list_logical_devices('GPU')
print(len(gpus), "Physical GPUs,", len(log_gpus), "Logical GPU")

from psutil import *

!lscpu|grep "Model name"
!nvidia-smi -L

In [ ]:
def split_input_target(sequence):
    sequence = list(sequence)
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return {"input_seq": input_text, "output_seq": target_text}

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_object_ = tf.keras.losses.SparseCategoricalCrossentropy(
      from_logits=True, 
      reduction="none"
      ) 
    loss_ = loss_object_(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)

def create_sequences(df):
    df["item_code"] = df["item_code"].astype("string")
    df["recency"] = df["recency"].astype("string")
    df["payment"] = df["payment"].astype("string")
    df["category_code"] = df["category_code"].astype("string")
    df["method_code"] = df["method_code"].astype("string")

    g_df = df.groupby("c_id", as_index=False).agg(
          {"item_code": lambda x: ", ".join(x),
          "recency": lambda y: ", ".join(y),
          "payment": lambda z: ", ".join(z),
          "category_code": lambda t: ", ".join(t),
          "method_code": lambda u: ", ".join(u)}
          )
  
    g_df["item_code"] = g_df["item_code"].apply(
      lambda x: x.split(", ")
      )
    g_df["recency"] = g_df["recency"].apply(
      lambda y: y.split(", ")
      )
    g_df["payment"] = g_df["payment"].apply(
      lambda z: z.split(", ")
      )
    g_df["category_code"] = g_df["category_code"].apply(
      lambda t: t.split(", ")
      )
    g_df["method_code"] = g_df["method_code"].apply(
      lambda u: u.split(", ")
      )
    return g_df

def create_splits(g_df):
    s_df = g_df.merge(g_df["item_code"].apply(
      lambda w: pd.Series(split_input_target(w))
      ), left_index=True, right_index=True)
    s_df["recency"] = s_df["recency"].apply(lambda p: list(p)[:-1])
    s_df["payment"] = s_df["payment"].apply(lambda q: list(q)[:-1])
    s_df["category_code"] = s_df["category_code"].apply(lambda r: list(r)[:-1])
    s_df["method_code"] = s_df["method_code"].apply(lambda s: list(s)[:-1])
    
    return s_df

def make_padding(df, n_padded):
    input_seq_padded = keras.preprocessing.sequence.pad_sequences(
        df["input_seq"].values, 
        maxlen=n_padded,
        padding="pre",
        value=0.0
    )
    recency_seq_padded = keras.preprocessing.sequence.pad_sequences(
        df["recency"].values,
        maxlen=n_padded,
        padding="pre",
        value=0.0
    )
    payment_seq_padded = keras.preprocessing.sequence.pad_sequences(
        df["payment"].values,
        maxlen=n_padded,
        padding="pre",
        value=0.0
    )
    category_seq_padded = keras.preprocessing.sequence.pad_sequences(
        df["category_code"].values,
        maxlen=n_padded,
        padding="pre",
        value=0.0
    )
    method_seq_padded = keras.preprocessing.sequence.pad_sequences(
        df["method_code"].values,
        maxlen=n_padded,
        padding="pre",
        value=0.0
    )
    if len(df.columns[df.columns.str.contains("output_seq")]) > 0:
        output_seq_padded = keras.preprocessing.sequence.pad_sequences(
            df["output_seq"].values,
            maxlen=n_padded,
            padding="pre",
            value=0.0
        )
    else:
        output_seq_padded = []
    return [input_seq_padded,
            recency_seq_padded,
            payment_seq_padded,
            category_seq_padded,
            method_seq_padded,
            output_seq_padded]

In [ ]:
def model1(hparams, n_in, n_emb, n_out,
           loss_function=loss_function):
    input_item = layers.Input(batch_input_shape=[None, n_in],
                              dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=n_emb,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_item)                              
    lstm = layers.LSTM(units=hparams[LSTM_UNITS],
                       return_sequences=True)(embedding_item)
    lstm = layers.Dropout(rate=hparams[DROP_RATE])(lstm)
    lstm = layers.BatchNormalization()(lstm)
    att = layers.Attention(use_scale=False,
                         causal=True)(inputs=[lstm, lstm],
                                      mask=[encoding_padding_mask,
                                            encoding_padding_mask])
    output = layers.Dense(n_out)(att)
    model = keras.Model(input_item, output)
    model.compile(loss=loss_function,
                  optimizer=keras.optimizers.Adam(learning_rate=hparams[L_RATE]),
                  metrics=["sparse_categorical_accuracy"])
    return model

def model2(hparams, n_in, n_emb1, n_emb2, n_out,
           loss_function=loss_function):
    input_item = layers.Input(batch_input_shape=[None, n_in],
                              dtype=tf.int32)
    input_rec = layers.Input(batch_input_shape=[None, n_in],
                             dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=n_emb1,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_item)  
    embedding_rec = layers.Embedding(input_dim=n_emb2,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_rec)
    concat_inputs = layers.Concatenate()([embedding_item, embedding_rec])
    concat_inputs = layers.BatchNormalization()(concat_inputs)
    lstm = layers.LSTM(units=hparams[LSTM_UNITS],
                       return_sequences=True)(concat_inputs)
    lstm = layers.Dropout(rate=hparams[DROP_RATE])(lstm)
    lstm = layers.BatchNormalization()(lstm)
    att = layers.Attention(use_scale=False,
                         causal=True)(inputs=[lstm, lstm],
                                      mask=[encoding_padding_mask,
                                            encoding_padding_mask])
    output = layers.Dense(n_out)(att)
    model = keras.Model([input_item, input_rec], output)
    model.compile(loss=loss_function,
                  optimizer=keras.optimizers.Adam(learning_rate=hparams[L_RATE]),
                  metrics=["sparse_categorical_accuracy"])
    return model

def model3(hparams, n_in, n_emb1, n_emb2, n_emb3, n_out,
           loss_function=loss_function):
    input_item = layers.Input(batch_input_shape=[None, n_in],
                              dtype=tf.int32)
    input_rec = layers.Input(batch_input_shape=[None, n_in],
                             dtype=tf.int32)
    input_pay = layers.Input(batch_input_shape=[None, n_in],
                             dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=n_emb1,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_item)  
    embedding_rec = layers.Embedding(input_dim=n_emb2,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_rec)
    embedding_pay = layers.Embedding(input_dim=n_emb3,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_pay)
    concat_inputs = layers.Concatenate()([embedding_item,
                                          embedding_rec,
                                          embedding_pay])
    concat_inputs = layers.BatchNormalization()(concat_inputs)
    lstm = layers.LSTM(units=hparams[LSTM_UNITS],
                       return_sequences=True)(concat_inputs)
    lstm = layers.Dropout(rate=hparams[DROP_RATE])(lstm)
    lstm = layers.BatchNormalization()(lstm)
    att = layers.Attention(use_scale=False,
                         causal=True)(inputs=[lstm, lstm],
                                      mask=[encoding_padding_mask,
                                            encoding_padding_mask])
    output = layers.Dense(n_out)(att)
    model = keras.Model([input_item, input_rec, input_pay], output)
    model.compile(loss=loss_function,
                  optimizer=keras.optimizers.Adam(learning_rate=hparams[L_RATE]),
                  metrics=["sparse_categorical_accuracy"])
    return model

def model4(hparams, n_in, n_emb1, n_emb2, n_emb3, n_out,
           loss_function=loss_function):
    input_item = layers.Input(batch_input_shape=[None, n_in],
                              dtype=tf.int32)
    input_rec = layers.Input(batch_input_shape=[None, n_in],
                             dtype=tf.int32)
    input_cat = layers.Input(batch_input_shape=[None, n_in],
                             dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=n_emb1,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_item)  
    embedding_rec = layers.Embedding(input_dim=n_emb2,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_rec)
    embedding_cat = layers.Embedding(input_dim=n_emb3,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_cat)
    concat_inputs = layers.Concatenate()([embedding_item,
                                          embedding_rec,
                                          embedding_cat])
    concat_inputs = layers.BatchNormalization()(concat_inputs)
    lstm = layers.LSTM(units=hparams[LSTM_UNITS],
                       return_sequences=True)(concat_inputs)
    lstm = layers.Dropout(rate=hparams[DROP_RATE])(lstm)
    lstm = layers.BatchNormalization()(lstm)
    att = layers.Attention(use_scale=False,
                         causal=True)(inputs=[lstm, lstm],
                                      mask=[encoding_padding_mask,
                                            encoding_padding_mask])
    output = layers.Dense(n_out)(att)
    model = keras.Model([input_item, input_rec, input_cat], output)
    model.compile(loss=loss_function,
                  optimizer=keras.optimizers.Adam(learning_rate=hparams[L_RATE]),
                  metrics=["sparse_categorical_accuracy"])
    return model

def model5(hparams, n_in, n_emb1, n_emb2, n_emb3, n_out,
           loss_function=loss_function):
    input_item = layers.Input(batch_input_shape=[None, n_in],
                              dtype=tf.int32)
    input_rec = layers.Input(batch_input_shape=[None, n_in],
                             dtype=tf.int32)
    input_met = layers.Input(batch_input_shape=[None, n_in],
                             dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=n_emb1,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_item)  
    embedding_rec = layers.Embedding(input_dim=n_emb2,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_rec)
    embedding_met = layers.Embedding(input_dim=n_emb3,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_met)
    concat_inputs = layers.Concatenate()([embedding_item,
                                          embedding_rec,
                                          embedding_met])
    concat_inputs = layers.BatchNormalization()(concat_inputs)
    lstm = layers.LSTM(units=hparams[LSTM_UNITS],
                       return_sequences=True)(concat_inputs)
    lstm = layers.Dropout(rate=hparams[DROP_RATE])(lstm)
    lstm = layers.BatchNormalization()(lstm)
    att = layers.Attention(use_scale=False,
                         causal=True)(inputs=[lstm, lstm],
                                      mask=[encoding_padding_mask,
                                            encoding_padding_mask])
    output = layers.Dense(n_out)(att)
    model = keras.Model([input_item, input_rec, input_met], output)
    model.compile(loss=loss_function,
                  optimizer=keras.optimizers.Adam(learning_rate=hparams[L_RATE]),
                  metrics=["sparse_categorical_accuracy"])
    return model

def model6(hparams, n_in, n_emb1, n_emb2, n_emb3, n_emb4, n_out,
           loss_function=loss_function):
    input_item = layers.Input(batch_input_shape=[None, n_in],
                              dtype=tf.int32)
    input_rec = layers.Input(batch_input_shape=[None, n_in],
                             dtype=tf.int32)
    input_pay = layers.Input(batch_input_shape=[None, n_in],
                             dtype=tf.int32)
    input_cat = layers.Input(batch_input_shape=[None, n_in],
                             dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=n_emb1,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_item)  
    embedding_rec = layers.Embedding(input_dim=n_emb2,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_rec)
    embedding_pay = layers.Embedding(input_dim=n_emb3,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_pay)
    embedding_cat = layers.Embedding(input_dim=n_emb4,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_cat)
    concat_inputs = layers.Concatenate()([embedding_item,
                                          embedding_rec,
                                          embedding_pay,
                                          embedding_cat])
    concat_inputs = layers.BatchNormalization()(concat_inputs)
    lstm = layers.LSTM(units=hparams[LSTM_UNITS],
                       return_sequences=True)(concat_inputs)
    lstm = layers.Dropout(rate=hparams[DROP_RATE])(lstm)
    lstm = layers.BatchNormalization()(lstm)
    att = layers.Attention(use_scale=False,
                         causal=True)(inputs=[lstm, lstm],
                                      mask=[encoding_padding_mask,
                                            encoding_padding_mask])
    output = layers.Dense(n_out)(att)
    model = keras.Model([input_item,
                         input_rec,
                         input_pay,
                         input_cat], output)
    model.compile(loss=loss_function,
                  optimizer=keras.optimizers.Adam(learning_rate=hparams[L_RATE]),
                  metrics=["sparse_categorical_accuracy"])
    return model

def model7(hparams, n_in, n_emb1, n_emb2, n_emb3, n_emb4,
           n_out, loss_function=loss_function):
    input_item = layers.Input(batch_input_shape=[None, n_in],
                              dtype=tf.int32)
    input_rec = layers.Input(batch_input_shape=[None, n_in],
                             dtype=tf.int32)
    input_cat = layers.Input(batch_input_shape=[None, n_in],
                             dtype=tf.int32)
    input_met = layers.Input(batch_input_shape=[None, n_in],
                             dtype=tf.int32)
    encoding_padding_mask = tf.math.logical_not(tf.math.equal(input_item, 0))
    embedding_item = layers.Embedding(input_dim=n_emb1,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_item)  
    embedding_rec = layers.Embedding(input_dim=n_emb2,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_rec)
    embedding_cat = layers.Embedding(input_dim=n_emb3,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_cat)
    embedding_met = layers.Embedding(input_dim=n_emb4,
                                      output_dim=hparams[EMB_UNITS]
                                     )(input_met)
    concat_inputs = layers.Concatenate()([embedding_item,
                                          embedding_rec,
                                          embedding_cat,
                                          embedding_met])
    concat_inputs = layers.BatchNormalization()(concat_inputs)
    lstm = layers.LSTM(units=hparams[LSTM_UNITS],
                       return_sequences=True)(concat_inputs)
    lstm = layers.Dropout(rate=hparams[DROP_RATE])(lstm)
    lstm = layers.BatchNormalization()(lstm)
    att = layers.Attention(use_scale=False,
                         causal=True)(inputs=[lstm, lstm],
                                      mask=[encoding_padding_mask,
                                            encoding_padding_mask])
    output = layers.Dense(n_out)(att)
    model = keras.Model([input_item,
                         input_rec,
                         input_cat,
                         input_met], output)
    model.compile(loss=loss_function,
                  optimizer=keras.optimizers.Adam(learning_rate=hparams[L_RATE]),
                  metrics=["sparse_categorical_accuracy"])
    return model

In [ ]:
data = pd.read_csv(
    "../input/ecommerce/time_train_data.csv",
     sep=";")

In [ ]:
df = data.copy()

In [ ]:
pro_max = df["item_code"].astype("int64").max()
rec_max = df["recency"].astype("int64").max()
pay_max = df["payment"].astype("int64").max()
cat_max = df["category_code"].astype("int64").max()
met_max = df["method_code"].astype("int64").max()
n_padded = 12
batch_size = 128

In [ ]:
def model_training(model, train_x, train_y,
                   batch_size=batch_size):
  
    es = keras.callbacks.EarlyStopping(monitor="val_loss",
                                       mode="min",
                                       verbose=1,
                                       patience=3)
    
    hist = model.fit(train_x, train_y,
                     batch_size=batch_size,
                     epochs=200,
                     verbose=0,
                     validation_split=0.1,
                     callbacks=[es])
    return hist

In [ ]:
EMB_UNITS = hp.HParam("emb_units", hp.Discrete([32, 64, 96, 128]))
LSTM_UNITS = hp.HParam("lstm_units", hp.Discrete([100, 200, 300, 400, 500, 600]))
DROP_RATE = hp.HParam("dropout", hp.Discrete([0.6, 0.8]))
L_RATE = hp.HParam("l_rate", hp.Discrete([0.001]))
REPLICATION = hp.HParam("replication", hp.Discrete([1, 2, 3]))
METRIC_ACCURACY = "sparse_categorical_accuracy"

In [ ]:
ldir = "logs/model7/tune/"
model_name = "model7"
with tf.summary.create_file_writer(ldir).as_default():
    hp.hparams_config(
          hparams=[EMB_UNITS, LSTM_UNITS, DROP_RATE,
                   L_RATE, REPLICATION],
          metrics=[hp.Metric(METRIC_ACCURACY, display_name="sparse_categorical_accuracy")])

train_data = create_sequences(df)
train_data = create_splits(train_data)
train_list = make_padding(train_data, n_padded)

In [ ]:
def run(run_dir, hparams, model):
    with tf.summary.create_file_writer(run_dir).as_default():
        hp.hparams(hparams)
        hist = model_training(model,
                              [train_list[0], train_list[1],
                               train_list[3], train_list[4]],
                              train_list[-1],
                              batch_size)
        m = hist.history["val_sparse_categorical_accuracy"][-1]
        print("m:",m)  
        tf.summary.scalar(METRIC_ACCURACY, m, step=2)

In [ ]:
session_num = 0

for e in EMB_UNITS.domain.values:
    for l in LSTM_UNITS.domain.values:
        for d in DROP_RATE.domain.values:
            for o in L_RATE.domain.values:
                for r in REPLICATION.domain.values:
                    hparams = {
                        EMB_UNITS: e,
                        LSTM_UNITS: l,
                        DROP_RATE: d,
                        L_RATE: o,
                        REPLICATION: r
                    }
                    model = model7(hparams, n_padded,
                                   pro_max+1, rec_max+1,
                                   cat_max+1, met_max+1, pro_max+1)
                    run_name = model_name+"-"+str(session_num)+"-"+str(r)
                    print("Experiment: %s" % run_name)
                    print({h.name: hparams[h] for h in hparams})
                    run(ldir + run_name, hparams, model)
                session_num += 1

Experiment: model7-0-1
{'emb_units': 32, 'lstm_units': 100, 'dropout': 0.6, 'l_rate': 0.001, 'replication': 1}


2021-10-25 13:09:30.612791: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:185] None of the MLIR Optimization Passes are enabled (registered 2)
2021-10-25 13:09:33.904601: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:689] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla P100-PCIE-16GB" frequency: 1328 num_cores: 56 environment { key: "architecture" value: "6.0" } environment { key: "cuda" value: "11000" } environment { key: "cudnn" value: "8005" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 4194304 shared_memory_size_per_multiprocessor: 65536 memory_size: 16152002560 bandwidth: 732160000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }
2021-10-25 13:09:35.274128: I tensorflow/stream_executor/cuda/cuda_dnn.cc:369] Loaded cuDNN version 8005
2021-10-25 13:09:39.345614: W tensorflow/core/grap

Epoch 00014: early stopping
m: 0.38531848788261414
Experiment: model7-0-2
{'emb_units': 32, 'lstm_units': 100, 'dropout': 0.6, 'l_rate': 0.001, 'replication': 2}


2021-10-25 13:10:18.125945: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:689] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla P100-PCIE-16GB" frequency: 1328 num_cores: 56 environment { key: "architecture" value: "6.0" } environment { key: "cuda" value: "11000" } environment { key: "cudnn" value: "8005" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 4194304 shared_memory_size_per_multiprocessor: 65536 memory_size: 16152002560 bandwidth: 732160000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }
2021-10-25 13:10:20.891048: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:689] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla P100-PCIE-16GB" fr

Epoch 00014: early stopping
m: 0.4001713693141937
Experiment: model7-0-3
{'emb_units': 32, 'lstm_units': 100, 'dropout': 0.6, 'l_rate': 0.001, 'replication': 3}


...

Epoch 00013: early stopping
m: 0.37532132863998413
Experiment: model7-47-2
{'emb_units': 128, 'lstm_units': 600, 'dropout': 0.8, 'l_rate': 0.001, 'replication': 2}


2021-10-25 14:48:31.602352: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:689] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla P100-PCIE-16GB" frequency: 1328 num_cores: 56 environment { key: "architecture" value: "6.0" } environment { key: "cuda" value: "11000" } environment { key: "cudnn" value: "8005" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 4194304 shared_memory_size_per_multiprocessor: 65536 memory_size: 16152002560 bandwidth: 732160000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }
2021-10-25 14:48:34.817996: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:689] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla P100-PCIE-16GB" fr

Epoch 00013: early stopping
m: 0.357326477766037
Experiment: model7-47-3
{'emb_units': 128, 'lstm_units': 600, 'dropout': 0.8, 'l_rate': 0.001, 'replication': 3}


2021-10-25 14:49:16.115445: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:689] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla P100-PCIE-16GB" frequency: 1328 num_cores: 56 environment { key: "architecture" value: "6.0" } environment { key: "cuda" value: "11000" } environment { key: "cudnn" value: "8005" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 4194304 shared_memory_size_per_multiprocessor: 65536 memory_size: 16152002560 bandwidth: 732160000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }
2021-10-25 14:49:19.322264: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:689] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla P100-PCIE-16GB" fr

Epoch 00013: early stopping
m: 0.36332476139068604
